![NVIDIA Header](../../shared/assets/nvidia_header.png)

# What Is OpenShell?

### The sandbox runtime that keeps AI agents from going rogue

⏱ Estimated time: 30 minutes

## Learning Objectives

By the end of this notebook, you will be able to:

<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Explain what <a href="https://docs.nvidia.com/openshell/about/overview">OpenShell</a> is and where it fits in the NemoClaw stack</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Use every core <code>openshell</code> CLI command</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Create, inspect, execute commands in, and delete <a href="https://docs.nvidia.com/openshell/sandboxes/manage-sandboxes">sandboxes</a></p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Read sandbox <a href="https://docs.nvidia.com/openshell/observability/accessing-logs">logs</a> and understand the log format</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; View and apply sandbox <a href="https://docs.nvidia.com/openshell/reference/policy-schema">policies</a></p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; List and manage <a href="https://docs.nvidia.com/openshell/sandboxes/manage-providers">inference providers</a></p>

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2024-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

## Setup

**Platform:** Validated NemoClaw host (Ubuntu 24.04, DGX Spark, or Apple Silicon macOS with Colima)

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd()
repo_root = _cwd
for _parent in [_cwd, *_cwd.parents]:
    if (_parent / "pyproject.toml").exists():
        repo_root = _parent
        if str(_parent) not in sys.path:
            sys.path.insert(0, str(_parent))
        break

In [ ]:
from shared.utils.setup import setup

setup(
    require_docker=True,
    require_node=True,
    require_openshell=True,
    required_keys=[],
)

---

## 1. What Is OpenShell?

OpenShell is the **sandbox runtime** that creates isolated environments for AI agents and
enforces security policies on everything they do. Think of it as a container runtime
built for AI agent safety.

Without a sandbox, an AI agent has the same access as the user who launched it — it can
read any file, reach any endpoint, and run any program. OpenShell removes that implicit
trust and replaces it with explicit, auditable policy controls. See the
[OpenShell Overview](https://docs.nvidia.com/openshell/about/overview) for the design rationale.

When you run an AI agent inside an OpenShell sandbox:

- The agent **cannot access files** outside its allowed paths
- The agent **cannot reach network endpoints** unless the policy explicitly allows them
- The agent **runs as a non-root user** with restricted system calls
- Every denied action is **logged** so you can audit what happened

### Where OpenShell Fits in the Stack

```
┌─────────────────────────────────────────────┐
│  You (the operator)                         │
│  Type commands, read output, watch dashboard │
├─────────────────────────────────────────────┤
│  NemoClaw CLI                               │  ← Deploys and manages agents
│  nemoclaw onboard / connect / logs          │
├─────────────────────────────────────────────┤
│  OpenShell                                  │  ← THIS MODULE
│  Creates sandboxes, enforces policies       │  ← Isolation + enforcement
│  openshell sandbox / policy / logs          │
├─────────────────────────────────────────────┤
│  OpenClaw (inside the sandbox)              │  ← The AI agent
│  Personality, skills, memory, channels      │  ← Talks to models, uses tools
└─────────────────────────────────────────────┘
```

OpenShell is the **middle layer** — it doesn't know what the agent does, only what the
agent is **allowed** to do. The policies you write in this module define that boundary.
For how the layers communicate, see the
[Architecture documentation](https://docs.nvidia.com/openshell/about/architecture).

### Follow One Action Through Enforcement

![OpenShell enforcement lifecycle: an agent action is evaluated against filesystem, network, process, and inference policy, then explicitly allowed or denied with evidence](../../assets/diagrams/openshell-enforcement-lifecycle.png)

Read the diagram from left to right: the sandbox proposes one bounded action, OpenShell evaluates the declared authority, and both the allow and deny paths produce inspectable evidence. A denial is often the expected proof that the boundary works; it is not automatically a lab failure.

---

## 2. Basics: Version and Help

Start with the two commands that confirm OpenShell is installed and show what it can do.

In [ ]:
from shared.utils.display import run_command

# What version of OpenShell is installed?
run_command("openshell --version")

Inspect the complete OpenShell command tree after confirming the installed version. **Expected result:** the help text lists the supported sandbox, policy, provider, inference, and log commands without an error. This map matters because later labs use each command group at a different runtime boundary.

In [ ]:
# The full command tree — every openshell subcommand
run_command("openshell --help")

The CLI is organized into **sandbox commands** (create, exec, delete, logs, policy) and
**gateway commands** (status, inference, doctor). In this module, you'll use the sandbox
commands extensively.

---

## 3. Sandbox Lifecycle

A **sandbox** is an isolated environment where an agent runs. Each sandbox has its own
filesystem, network namespace, and security policy. That isolation is what keeps one
agent from interfering with another or touching host resources it shouldn't.
See [Manage Sandboxes](https://docs.nvidia.com/openshell/sandboxes/manage-sandboxes)
for the full lifecycle documentation.

### List Sandboxes

See what's currently running:

In [ ]:
# List all sandboxes -- shows name, namespace, creation time, and phase (Ready/Pending/Failed)
run_command("openshell sandbox list")

from shared.utils.platform import resolve_course_sandbox_name

course_sandbox_name = resolve_course_sandbox_name()
if not course_sandbox_name:
    raise RuntimeError(
        "No course sandbox found. Expected the standard course sandbox name `agent`. Older DGX Spark setups may still use `training-sandbox`; rerun `./setup.sh` from the repository root if neither exists."
    )
print(f"Using course sandbox: {course_sandbox_name}")

You should see at least one course sandbox. The standard course name is `agent`. Older DGX Spark setups may still show `training-sandbox` until you rerun `./setup.sh`. The **Phase** column tells you whether it is ready to use.

### Execute a Command Inside a Sandbox

The most important command — run something inside the isolated environment:


In [ ]:
# Who are you inside the sandbox? (Should be 'sandbox', not 'root')
run_command(f"openshell sandbox exec -n {course_sandbox_name} -- whoami")

Inspect only the teaching-relevant environment variables inside the course sandbox. **Expected result:** proxy and runtime variables are visible, while any token value is replaced with `REDACTED`. This matters because the sandbox needs routing context without exposing usable credentials.

In [ ]:
# What environment variables exist inside the sandbox?
# Show only the teaching-relevant proxy/runtime variables and redact secrets.
run_command(
    f'openshell sandbox exec -n {course_sandbox_name} -- sh -lc \'env | grep -E "^(OPENCLAW_|HTTP_|HTTPS_|NO_PROXY|http_proxy|https_proxy|no_proxy|OPENSHELL_SANDBOX=|NODE_OPTIONS=|SSL_CERT_FILE=|REQUESTS_CA_BUNDLE=|CURL_CA_BUNDLE=)" | sed -E "s/(TOKEN=).*/\\\\1REDACTED/; s/(OPENCLAW_GATEWAY_TOKEN=).*/\\\\1REDACTED/" | sort\'',
    timeout=15,
    check=True,
)

List the sandbox workspace after checking its runtime environment. **Expected result:** `/sandbox/` is readable and shows the files mounted for the agent. This matters because policy enforcement operates on this sandbox-visible filesystem rather than arbitrary host paths.

In [ ]:
# What's in the sandbox filesystem?
run_command(f"openshell sandbox exec -n {course_sandbox_name} -- ls -la /sandbox/", timeout=15)

**Key observations:**
- The user is `sandbox`, not `root` — privilege is dropped
- `https_proxy` and `http_proxy` point to `10.200.0.1:3128` — the egress proxy that enforces [network policy](https://docs.nvidia.com/openshell/reference/policy-schema)
- `/sandbox/` is the agent's working directory with `.openclaw` (OpenClaw runtime config) and `.nemoclaw` (NemoClaw state)

### Create a New Sandbox

Create a fresh sandbox to see the lifecycle in action. The `create_sandbox` helper runs the current
non-interactive OpenShell create flow, waits for Ready, and then confirms `sandbox exec` works.

In [ ]:
from shared.utils.display import create_sandbox

# Create a test sandbox with the baseline policy
# This runs: openshell sandbox create --name cli-demo ...
# Then polls until it reaches Ready and accepts sandbox exec
create_sandbox(
    "cli-demo",
    policy=str(repo_root / "tracks/02-openshell-runtime/policies/openshell-baseline.policy.yaml"),
)

Confirm that the new `cli-demo` sandbox joined the runtime inventory. **Expected result:** `openshell sandbox list` shows both the course sandbox and `cli-demo` as distinct entries. This matters because creation is not complete until the gateway can account for the new sandbox.

In [ ]:
# Now we should see two sandboxes
run_command("openshell sandbox list")

Execute a simple command inside `cli-demo` to prove it is usable. **Expected result:** the cell prints `Hello from inside the sandbox!`. This matters because an inventory entry alone does not prove that sandbox execution works.

In [ ]:
# Run a command in our new sandbox
run_command("openshell sandbox exec -n cli-demo -- echo 'Hello from inside the sandbox!'")

### Upload and Download Files

You can transfer files between the host and a sandbox:

In [ ]:
# Show the upload command syntax (we won't actually upload, just show the help)
run_command("openshell sandbox upload --help")

The `upload` command copies files from your host into a sandbox — this is how skills
(SKILL.md files) and configuration get deployed. You'll use it in Module 4 when
building real integrations.

### Delete a Sandbox

In [ ]:
from shared.utils.display import delete_sandbox

# Clean up our test sandbox
delete_sandbox("cli-demo")

Check the inventory again after deleting the temporary sandbox. **Expected result:** `cli-demo` is absent while the course sandbox remains available. This matters because every lab should leave shared runtime state clean for the next learner.

In [ ]:
# Back to the course sandbox inventory
run_command("openshell sandbox list")

---

## 4. Observability: Logs

Sandbox logs are your primary debugging tool. Every network decision, SSH connection,
and policy evaluation is logged. When something goes wrong in production — an agent
can't reach an endpoint, a file write is denied — the logs tell you what happened
and why. See [Accessing Logs](https://docs.nvidia.com/openshell/observability/accessing-logs)
for advanced filtering and streaming options.

In [ ]:
# View the last 5 minutes of logs from the course sandbox
run_command(f"openshell logs {course_sandbox_name} --since 5m")

### Reading the Log Format

Each log line follows this structure:

```
[timestamp] [component] [LEVEL] [module] message key=value key=value ...
```

| Field | Example | What it tells you |
|-------|---------|------------------|
| `timestamp` | `1776205346.291` | Unix timestamp (seconds since epoch) |
| `component` | `[gateway]` or `[sandbox]` | Which side generated the log |
| `LEVEL` | `INFO`, `WARN`, `ERROR` | Severity |
| `module` | `openshell_sandbox::proxy` | Which code module |
| `message` | `CONNECT action=deny` | What happened |
| `key=value` | `dst_host=httpbin.org` | Structured details |

**The most important log pattern:** `action=deny` — this means the policy blocked something.
Look for `dst_host`, `binary`, and `reason` to understand what was blocked and why.

---

## 5. Policy Management

Policies define what a sandbox can and cannot do. Every OpenShell security boundary
is enforced through a policy YAML file. Without a policy, a sandbox defaults to
permissive mode — which defeats the point of sandboxing. Writing precise policies is
the core skill you'll build in this module. See
[Policy Customization](https://docs.nvidia.com/openshell/sandboxes/policies) for the
full configuration guide.

### View the Current Policy

In [ ]:
# What policy is the course sandbox currently running?
run_command(f"openshell policy get {course_sandbox_name}")

Retrieve the full policy document, including its version and hash metadata. **Expected result:** OpenShell prints the active YAML policy plus identifying metadata. This matters because operators need an auditable policy identity, not just a summarized view.

In [ ]:
# The --full flag includes the YAML header with version and hash
run_command(f"openshell policy get {course_sandbox_name} --full")

### Incremental Policy Updates

Current OpenShell builds include `openshell policy update` for targeted runtime edits. Use it when you need to add or remove one network rule without replacing the whole policy document.

```bash
openshell policy update agent --add-endpoint api.github.com:443:read-only:rest:enforce
openshell policy update agent --add-allow 'api.github.com:443:GET:/repos/**'
```

Reserve `openshell policy set` for reviewed policy files. Use `openshell policy update` for narrow, auditable runtime changes.


In [ ]:
# Show the incremental policy update interface without changing the sandbox.
run_command("openshell policy update --help", timeout=15)

### Policy Structure

Every policy has four sections:

| Section | What it controls | Static or Dynamic? |
|---------|-----------------|-------------------|
| `filesystem_policy` | Which paths the sandbox can read/write | **Static** — locked at creation |
| `landlock` | Kernel-level [filesystem enforcement](https://docs.nvidia.com/openshell/reference/policy-schema) mode | **Static** — locked at creation |
| `process` | User identity (`sandbox` vs `root`) | **Static** — locked at creation |
| `network_policies` | Which endpoints the sandbox can reach | **Dynamic** — hot-reloadable |

Static sections can only be changed by recreating the sandbox. [Network policies](https://docs.nvidia.com/openshell/reference/policy-schema) can be
updated on a running sandbox with `openshell policy set`.

> 💡 **Deep dive:** Notebooks 02-07 in this module teach you how to write each policy
> section from scratch.

---

## 6. Inference Providers

Providers manage how AI model credentials reach the sandbox. The **[privacy router](https://docs.nvidia.com/openshell/inference/about)**
intercepts inference requests and injects real API keys so the agent never sees them.
If the agent is compromised, the attacker cannot steal your API keys — they never
existed inside the sandbox. See
[Manage Providers](https://docs.nvidia.com/openshell/sandboxes/manage-providers) for
setup details and
[Inference Configuration](https://docs.nvidia.com/openshell/inference/configure)
for routing options.

In [ ]:
# What inference providers are configured?
run_command("openshell provider list")

The provider was created when `setup.sh` ran NemoClaw onboarding. Inside the sandbox, the agent sees
a **placeholder token** (`openshell:resolve:env:NVIDIA_API_KEY`) instead of the real key.
When a request goes through the egress proxy, the privacy router swaps the placeholder
for the real credential — so a compromised agent cannot exfiltrate your keys, because
the real values never enter the sandbox.

---

## 7. Advanced Commands

These commands are used less often but matter for debugging and remote access. See
[Security Best Practices](https://docs.nvidia.com/openshell/security/best-practices)
for guidance on hardening remote sandbox access.

In [ ]:
# Gateway status — shows the OpenShell gateway health
run_command("openshell status")

Display the current `ssh-proxy` command shape used by OpenSSH `ProxyCommand`. **Expected result:** the example contains explicit gateway and sandbox placeholders. This matters because direct shell access still crosses the OpenShell gateway boundary.

In [ ]:
# SSH proxy — used by OpenSSH ProxyCommand for direct sandbox access.
# Show the current pattern without dumping legacy compatibility flags.
print("ProxyCommand example:")
print("openshell ssh-proxy --gateway <gateway-url> --sandbox <sandbox-id>")

| Command | When to use it |
|---------|---------------|
| `openshell status` | Check if the OpenShell gateway is healthy |
| `openshell ssh-proxy` | SSH directly into a sandbox (used by `ProxyCommand` in SSH config) |
| `openshell term` | Open the TUI for real-time network monitoring (interactive) |
| `openshell doctor` | Diagnose gateway issues |
| `openshell sandbox connect` | Interactive shell into a sandbox (like `nemoclaw <name> connect`) |

---

## 8. CLI Quick Reference

| Command | What it does |
|---------|-------------|
| `openshell --version` | Show installed version |
| `openshell sandbox list` | List all sandboxes |
| `openshell sandbox create --name N --policy P` | Create a sandbox |
| `openshell sandbox exec -n N -- CMD` | Run a command in a sandbox |
| `openshell sandbox delete N` | Delete a sandbox |
| `openshell sandbox upload N SRC DST` | Upload files to sandbox |
| `openshell sandbox download N SRC DST` | Download files from sandbox |
| `openshell sandbox connect N` | Interactive shell |
| `openshell logs N --since T` | View sandbox logs |
| `openshell logs N --follow` | Stream logs in real time |
| `openshell policy get N` | View sandbox policy |
| `openshell policy get N --full` | Full policy with YAML header |
| `openshell policy set N --policy F --wait` | Apply a policy |
| `openshell provider list` | List inference providers |
| `openshell provider create` | Create a provider |
| `openshell status` | Gateway health |
| `openshell term` | Interactive TUI |
| `openshell doctor` | Diagnose issues |

| `openshell policy update N --add-endpoint ...` | Add a targeted runtime policy rule |


---

## Real-World Connection

Every command you ran in this notebook works identically on a production server.
In a real deployment:

- `openshell sandbox list` shows all agent sandboxes running on a remote GPU machine
- `openshell logs` streams production deny events to your terminal or monitoring stack
- `openshell policy set` hot-reloads network policies without restarting agents
- `openshell provider create` configures inference credentials that never enter the sandbox

The local training environment uses the same OpenShell binaries, policies, and CLI as production.

---

## Checkpoint

Verify the OpenShell runtime before moving on.

### Command-Output Verification

1. **Check:** `openshell sandbox list` shows the course sandbox as Ready.
   - **Expected:** one row with Phase `Ready` for the sandbox that `setup.sh` created.
   - **If not working:** rerun the setup cell, or rerun `./setup.sh` from the repository root.

2. **Check:** `openshell provider list` shows a configured inference provider.
   - **Expected:** at least one provider row (for example `ollama-local` on DGX Spark or `nvidia-prod` on Brev).
   - **If not working:** reread the providers section of this notebook, then re-run `openshell provider list` inside the course checkout.

3. **Check:** the policy get output contains the four policy domains the notebook introduced.
   - **Expected:** the output shows `filesystem`, `network`, `process`, and `inference` sections.
   - **If not working:** rerun the policy-preview cell in this notebook; if it still fails, your sandbox is not Ready and you should rerun `./setup.sh`.


---

## ✅ What You Accomplished

<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Understood what OpenShell is and where it fits in the stack</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Used every core <code>openshell</code> CLI command with real output</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Created, inspected, and deleted a sandbox</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Read sandbox logs and understood the log format</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Viewed sandbox policies and understood the 4-section structure</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Listed inference providers and understood the privacy router</p>

<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Learned when to use incremental <code>openshell policy update</code> instead of replacing a full policy file</p>


## Next Steps

Now that you know what OpenShell can do, it's time to write policies. Continue to
**[Sandbox Policies](02_sandbox_policies.ipynb)** to write a complete security policy
and prove each domain against a live sandbox.

---

<small>

**Sources:**
[OpenShell Overview](https://docs.nvidia.com/openshell/about/overview) ·
[Architecture](https://docs.nvidia.com/openshell/about/architecture) ·
[Manage Sandboxes](https://docs.nvidia.com/openshell/sandboxes/manage-sandboxes) ·
[Manage Providers](https://docs.nvidia.com/openshell/sandboxes/manage-providers) ·
[Policy Customization](https://docs.nvidia.com/openshell/sandboxes/policies) ·
[Accessing Logs](https://docs.nvidia.com/openshell/observability/accessing-logs) ·
[GitHub](https://github.com/NVIDIA/OpenShell)

</small>